# Nebius Data Lab Fine-Tuning Tutorial: Customer Support Assistant

This notebook turns the `scripts/customer.py` workflow into a step-by-step tutorial. It shows how to:

1. generate baseline support interactions,
2. upload a raw dataset to Nebius Data Lab,
3. run batch inference with a stronger teacher model,
4. curate the outputs into a training set,
5. fine-tune a smaller student model with LoRA,
6. deploy the adapter as a private custom model, and
7. smoke test the deployed support assistant.

> This notebook mirrors the current `nebius-datalab-pipeline/scripts/customer.py` script, including the Data Lab payload shapes and the LoRA deployment logic.


## Prerequisites

- A Nebius API key exported as `NEBIUS_API_KEY`
- Python environment with `openai` and `requests` installed
- Permission to create Data Lab datasets, fine-tuning jobs, and private custom models

This notebook writes intermediate artifacts to `nebius-datalab-pipeline/scripts/artifacts/` and saves lightweight state to `customer_notebook_state.json` so you can rerun later steps without losing IDs.


In [ ]:
# Uncomment if this kernel still needs dependencies.
# %pip install -q openai requests


In [ ]:
import json
import os
import time
import uuid
from pathlib import Path
from typing import Any
from urllib.parse import quote

import requests
from openai import OpenAI

def locate_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "nebius-datalab-pipeline" / "scripts" / "customer.py").exists():
            return base
    return cwd

REPO_ROOT = locate_repo_root()
PIPELINE_DIR = REPO_ROOT / "nebius-datalab-pipeline"
ARTIFACT_DIR = PIPELINE_DIR / "scripts" / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = ARTIFACT_DIR / "customer_notebook_state.json"

API_KEY = os.environ["NEBIUS_API_KEY"]
BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
CONTROL_URL = "https://api.tokenfactory.nebius.com"
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
DEPLOY_BASE_MODEL_CANDIDATES = [
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    BASE_MODEL,
]
TEACHER_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
LORA_RANK = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
N_EPOCHS = 2
POLL_SECONDS = 30
DATA_LAB_FOLDER = "/demo/customer-support"
BATCH_COMPLETION_WINDOW = "24h"

SUPPORT_SYSTEM_PROMPT = (
    "You are a helpful customer support assistant for Northwind Gadgets. "
    "Policies: 30-day returns for unused physical items; damaged items "
    "reported within 7 days can be replaced; refunds go to the original "
    "payment method in 5-7 business days after approval; digital gift cards "
    "are non-refundable; subscription cancellations take effect at the end "
    "of the current billing cycle; shipping addresses can be changed only "
    "before fulfillment; warranty claims require the product serial number "
    "and proof of purchase; expedited shipping fees are non-refundable once "
    "an order has shipped."
)

TEACHER_SYSTEM_PROMPT = (
    "You are a senior customer support policy expert. "
    "Answer clearly, empathetically, and accurately. "
    "Never promise exceptions to policy. Ask only for the minimum extra "
    "information required to proceed."
)

DOMAIN_TOPICS = [
    "My headphones arrived yesterday and the right earcup is cracked. What can you do?",
    "I bought a travel backpack 12 days ago and never used it. Can I return it for a refund?",
    "I accidentally sent a digital gift card to the wrong email. Can you refund it?",
    "Can I change the shipping address on my order if it still says preparing for shipment?",
    "Please cancel my premium support subscription today. I do not want to be billed again.",
    "Tracking shows my return was delivered four days ago, but I still do not see the refund.",
    "My espresso machine stopped heating after two months. Can I file a warranty claim?",
    "I paid for expedited shipping, but I want that charge refunded after the order shipped.",
    "My monitor already shipped this morning. Can you reroute it to my office instead?",
    "My refund was approved, but the card I used is closed. Can you send it to a new card?",
]

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
HEADERS = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

print("Repo root:", REPO_ROOT)
print("Artifacts:", ARTIFACT_DIR)
print("Notebook state:", STATE_PATH)
print("Support topics:", len(DOMAIN_TOPICS))


## Helper functions

The next cell defines the same utility functions the script relies on: Data Lab requests, state persistence, response extraction, batch polling, fine-tuning, and deployment.

> Important: the fine-tuning call uses the current LoRA fields (`lora`, `lora_r`, `lora_alpha`, `lora_dropout`). Older field names such as `lora_rank` do not produce a deployable LoRA job.


In [ ]:
def load_state() -> dict[str, Any]:
    if not STATE_PATH.exists():
        return {}
    return json.loads(STATE_PATH.read_text())

def save_state(state: dict[str, Any]) -> None:
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True) + "\n")

def load_id_map_from_artifact() -> dict[str, str]:
    raw_dataset_path = ARTIFACT_DIR / "customer_raw_dataset.jsonl"
    if not raw_dataset_path.exists():
        raise RuntimeError(f"Missing {raw_dataset_path}. Re-run Step 1 and Step 2 first.")

    id_map: dict[str, str] = {}
    with raw_dataset_path.open() as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            custom_id = record.get("custom_id")
            prompt = record.get("prompt")
            if isinstance(custom_id, str) and isinstance(prompt, str):
                id_map[custom_id] = prompt
    return id_map

def datalab_request(method: str, path: str, *, json_body: dict[str, Any] | None = None, params: dict[str, Any] | None = None) -> requests.Response:
    response = requests.request(
        method,
        f"{CONTROL_URL}{path}",
        headers=HEADERS,
        json=json_body,
        params=params,
        timeout=60,
    )
    response.raise_for_status()
    return response

def teacher_messages(prompt: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": TEACHER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

def extract_text_content(value: Any) -> str:
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        parts = [extract_text_content(item) for item in value]
        return "\n".join([part for part in parts if part]).strip()
    if isinstance(value, dict):
        if isinstance(value.get("text"), str):
            return value["text"].strip()
        for key in ("content", "message"):
            text = extract_text_content(value.get(key))
            if text:
                return text
    return ""

def extract_prompt_from_record(record: dict[str, Any], id_map: dict[str, str]) -> str:
    custom_id = str(record.get("custom_id") or "").strip()
    if custom_id and custom_id in id_map:
        return id_map[custom_id]

    for key in ("messages", "completed_dialogue", "prompt"):
        value = record.get(key)
        if isinstance(value, list):
            for message in value:
                if isinstance(message, dict) and message.get("role") == "user":
                    text = extract_text_content(message.get("content"))
                    if text:
                        return text
        elif isinstance(value, str) and value.strip():
            return value.strip()

    return ""

def extract_reply_from_record(record: dict[str, Any]) -> str:
    for key in ("completion", "assistant_response", "output_text", "text"):
        text = extract_text_content(record.get(key))
        if text:
            return text

    response = record.get("response")
    if isinstance(response, dict):
        body = response.get("body") if isinstance(response.get("body"), dict) else response
        choices = body.get("choices") if isinstance(body, dict) else None
        if isinstance(choices, list) and choices:
            choice = choices[0]
            if isinstance(choice, dict):
                message = choice.get("message") if isinstance(choice.get("message"), dict) else None
                if message:
                    text = extract_text_content(message.get("content"))
                    if text:
                        return text
                text = extract_text_content(choice.get("text"))
                if text:
                    return text

        output = body.get("output") if isinstance(body, dict) else None
        if isinstance(output, list):
            for item in output:
                if isinstance(item, dict):
                    text = extract_text_content(item.get("content"))
                    if text:
                        return text

    for key in ("completed_dialogue", "messages"):
        value = record.get(key)
        if isinstance(value, list):
            assistant_messages = [
                extract_text_content(message.get("content"))
                for message in value
                if isinstance(message, dict) and message.get("role") == "assistant"
            ]
            assistant_messages = [msg for msg in assistant_messages if msg]
            if assistant_messages:
                return assistant_messages[-1]

    return ""

def deployment_base_models(ft_job_id: str) -> list[str]:
    candidates: list[str] = []
    try:
        job = client.fine_tuning.jobs.retrieve(ft_job_id)
        job_model = getattr(job, "model", None)
        if isinstance(job_model, str) and job_model and job_model not in candidates:
            candidates.append(job_model)
    except Exception as exc:
        print(f"Warning: could not retrieve fine-tuning job model: {exc}")

    for model_name in DEPLOY_BASE_MODEL_CANDIDATES:
        if model_name not in candidates:
            candidates.append(model_name)
    return candidates

def step1_generate_inference_logs(topics: list[str]) -> list[dict[str, Any]]:
    logs: list[dict[str, Any]] = []
    for topic in topics:
        resp = client.chat.completions.create(
            model=BASE_MODEL,
            messages=[
                {"role": "system", "content": SUPPORT_SYSTEM_PROMPT},
                {"role": "user", "content": topic},
            ],
            max_tokens=256,
        )
        logs.append({
            "prompt": topic,
            "completion": resp.choices[0].message.content,
            "model": BASE_MODEL,
        })
        print(f"✓ {topic[:70]}")
    return logs

def step2_upload_raw_dataset(logs: list[dict[str, Any]]) -> tuple[str, str, dict[str, str]]:
    path = ARTIFACT_DIR / "customer_raw_dataset.jsonl"
    dataset_name = f"customer-support-raw-{uuid.uuid4().hex[:8]}"
    rows: list[dict[str, Any]] = []
    id_map: dict[str, str] = {}

    with path.open("w") as handle:
        for entry in logs:
            record = {
                "custom_id": uuid.uuid4().hex,
                "prompt": entry["prompt"],
                "messages": teacher_messages(entry["prompt"]),
                "base_completion": entry["completion"] or "",
                "base_model": entry["model"],
            }
            handle.write(json.dumps(record) + "\n")
            rows.append(record)
            id_map[record["custom_id"]] = entry["prompt"]

    payload = {
        "name": dataset_name,
        "schema": [
            {"name": "custom_id", "type": {"name": "string"}},
            {"name": "prompt", "type": {"name": "string"}},
            {"name": "messages", "type": {"name": "json"}},
            {"name": "base_completion", "type": {"name": "string"}},
            {"name": "base_model", "type": {"name": "string"}},
        ],
        "folder": DATA_LAB_FOLDER,
        "rows": rows,
    }
    dataset = datalab_request("POST", "/v1/datasets", json_body=payload).json()
    print("Raw dataset ID:", dataset["id"])
    print("Raw dataset version:", dataset["current_version"])
    return dataset["id"], dataset["current_version"], id_map

def step3_run_batch_inference(source_dataset_id: str, source_dataset_version: str) -> tuple[str, str]:
    payload = {
        "type": "batch_inference",
        "src": [
            {
                "id": source_dataset_id,
                "version": source_dataset_version,
                "mapping": {
                    "type": "text_messages",
                    "messages": {"type": "column", "name": "messages"},
                    "custom_id": {"type": "column", "name": "custom_id"},
                    "max_tokens": {"type": "text", "value": "1024"},
                },
            }
        ],
        "dst": [],
        "params": {
            "model": TEACHER_MODEL,
            "completion_window": BATCH_COMPLETION_WINDOW,
        },
    }
    operation = datalab_request("POST", "/v1/operations", json_body=payload).json()
    if not operation.get("dst"):
        raise RuntimeError(f"No destination dataset returned: {json.dumps(operation, indent=2)}")

    dst_dataset_id = operation["dst"][0]["id"]
    print(f"Operation: {operation['id']} status={operation['status']}")
    print("Output dataset ID:", dst_dataset_id)

    while True:
        time.sleep(POLL_SECONDS)
        operation = datalab_request("GET", f"/v1/operations/{operation['id']}").json()
        print("Batch status:", operation["status"])
        if operation["status"] in {"succeeded", "failed", "cancelled", "unknown"}:
            break

    if operation["status"] != "succeeded":
        raise RuntimeError(f"Batch inference ended with status={operation['status']}")
    return operation["id"], dst_dataset_id

def step4_download_and_curate(output_dataset_id: str, id_map: dict[str, str]) -> Path:
    export_response = datalab_request(
        "GET",
        f"/v1/datasets/{output_dataset_id}/export",
        params={"format": "jsonl"},
    )

    raw_export_path = ARTIFACT_DIR / "customer_batch_output.jsonl"
    raw_export_path.write_text(export_response.text)
    output_path = ARTIFACT_DIR / "customer_curated_training.jsonl"
    records = [json.loads(line) for line in export_response.text.strip().splitlines() if line.strip()]

    written = 0
    with output_path.open("w") as handle:
        for rec in records:
            prompt = extract_prompt_from_record(rec, id_map)
            reply = extract_reply_from_record(rec)
            if not prompt or not reply or len(reply) < 50:
                continue
            sample = {
                "messages": [
                    {"role": "system", "content": SUPPORT_SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                    {"role": "assistant", "content": reply},
                ]
            }
            handle.write(json.dumps(sample) + "\n")
            written += 1

    if not records:
        raise RuntimeError(f"Batch output dataset {output_dataset_id} exported no rows.")
    if written == 0:
        raise RuntimeError(
            "Batch output export succeeded, but no usable assistant replies were found. "
            f"Inspect {raw_export_path} for the returned schema."
        )
    print(f"Curated {written} examples -> {output_path}")
    return output_path

def step5_upload_training_file(curated_path: Path) -> str:
    with curated_path.open("rb") as handle:
        file_obj = client.files.create(file=handle, purpose="fine-tune")
    print("Training file ID:", file_obj.id)
    return file_obj.id

def step6_launch_finetuning(train_file_id: str) -> str:
    job = client.fine_tuning.jobs.create(
        model=BASE_MODEL,
        training_file=train_file_id,
        hyperparameters={
            "learning_rate": 2e-5,
            "n_epochs": N_EPOCHS,
            "lora": True,
            "lora_r": LORA_RANK,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "packing": True,
        },
    )
    print(f"Job ID: {job.id} status={job.status}")

    while True:
        time.sleep(POLL_SECONDS)
        job = client.fine_tuning.jobs.retrieve(job.id)
        print(f"Fine-tune status={job.status} trained_tokens={job.trained_tokens}")
        if job.status in ("succeeded", "failed", "cancelled"):
            break

    if job.status != "succeeded":
        raise RuntimeError(f"Fine-tuning failed: {getattr(job, 'error', '')}")
    return job.id

def step7_deploy_lora(ft_job_id: str, adapter_name: str) -> dict[str, Any]:
    checkpoints = client.fine_tuning.jobs.checkpoints.list(ft_job_id).data
    if not checkpoints:
        raise RuntimeError("No checkpoints found for the fine-tuning job.")

    ckpt_id = checkpoints[-1].id
    attempt_errors: list[str] = []
    model_info: dict[str, Any] | None = None

    for base_model in deployment_base_models(ft_job_id):
        payload = {
            "source": f"{ft_job_id}:{ckpt_id}",
            "base_model": base_model,
            "name": adapter_name,
            "description": "Customer support Data Lab notebook demo model",
        }
        response = requests.post(f"{CONTROL_URL}/v0/models", json=payload, headers=HEADERS, timeout=60)
        if response.ok:
            model_info = response.json()
            print(f"Deploy accepted with base_model={base_model}")
            break

        error_body = response.text.strip() or "<empty body>"
        attempt_errors.append(
            f"base_model={base_model} status={response.status_code} body={error_body}"
        )
        print(f"Deploy rejected for base_model={base_model}: HTTP {response.status_code}")

    if model_info is None:
        raise RuntimeError(
            "LoRA deployment failed for all candidate base models:\n- " + "\n- ".join(attempt_errors)
        )

    model_name = model_info["name"]
    print("Deploy initiated:", model_name)

    for _ in range(60):
        time.sleep(10)
        encoded_name = quote(model_name, safe="")
        response = requests.get(f"{CONTROL_URL}/v0/models/{encoded_name}", headers=HEADERS, timeout=60)
        response.raise_for_status()
        info = response.json()
        status = info.get("status", "?")
        print("Deploy status:", status)
        if status == "active":
            return info
        if status == "error":
            raise RuntimeError(f"Deployment failed: {info.get('status_reason')}")

    raise TimeoutError("Model did not become active in time.")

def step8_smoke_test(model_full_name: str) -> list[dict[str, str]]:
    test_prompts = [
        "My package arrived damaged yesterday. What should I do?",
        "Can I return an unused backpack after 12 days?",
    ]
    answers = []
    for prompt in test_prompts:
        resp = client.chat.completions.create(
            model=model_full_name,
            messages=[
                {"role": "system", "content": SUPPORT_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            max_tokens=200,
        )
        answer = resp.choices[0].message.content
        answers.append({"prompt": prompt, "answer": answer})
        print(f"Q: {prompt}\nA: {answer}\n")
    return answers

state = load_state()
state


## Step 1 — Generate baseline customer-support interactions

We start with a smaller baseline model and domain-specific support prompts. These initial generations help seed the raw dataset and make the later teacher-student distillation flow more realistic than a generic prompt list.


In [ ]:
logs = step1_generate_inference_logs(DOMAIN_TOPICS)
state = load_state()
state.update({"last_completed_step": 1, "log_count": len(logs)})
save_state(state)
logs[:2]


## Step 2 — Upload the raw dataset to Data Lab

This stage stores a structured dataset in Data Lab with a stable dataset ID and version. Each row keeps a `custom_id`, the original prompt, teacher-ready `messages`, and the baseline completion for traceability.


In [ ]:
raw_dataset_id, raw_dataset_version, id_map = step2_upload_raw_dataset(logs)
state = load_state()
state.update({
    "raw_dataset_id": raw_dataset_id,
    "raw_dataset_version": raw_dataset_version,
    "last_completed_step": 2,
})
save_state(state)
{"raw_dataset_id": raw_dataset_id, "raw_dataset_version": raw_dataset_version}


## Step 3 — Run batch inference with the teacher model

The teacher model generates stronger answers over the uploaded dataset asynchronously. Expect the operation to spend some time in `queued` or `running` before it succeeds.


In [ ]:
operation_id, output_dataset_id = step3_run_batch_inference(raw_dataset_id, raw_dataset_version)
state = load_state()
state.update({
    "operation_id": operation_id,
    "output_dataset_id": output_dataset_id,
    "last_completed_step": 3,
})
save_state(state)
{"operation_id": operation_id, "output_dataset_id": output_dataset_id}


## Step 4 — Download, filter, and reformat the batch outputs

This is the highest-leverage step in the workflow. We export the teacher outputs, recover the original prompt for each row, discard weak replies, and write a conversational JSONL file ready for fine-tuning.


In [ ]:
if "id_map" not in globals():
    id_map = load_id_map_from_artifact()

curated_path = step4_download_and_curate(output_dataset_id, id_map)
state = load_state()
state.update({"curated_path": str(curated_path), "last_completed_step": 4})
save_state(state)
curated_path


## Step 5 — Upload the curated training file

The curated JSONL file is uploaded through the Nebius OpenAI-compatible Files API with `purpose="fine-tune"`. Save the returned file ID because it is the durable input to Step 6.


In [ ]:
training_file_id = step5_upload_training_file(curated_path)
state = load_state()
state.update({"training_file_id": training_file_id, "last_completed_step": 5})
save_state(state)
training_file_id


## Step 6 — Launch a LoRA fine-tuning job

This step creates the student model using LoRA. The important part is that the hyperparameters explicitly include `lora=True`, `lora_r`, `lora_alpha`, and `lora_dropout`; without them, you may end up with a job that is not deployable through the LoRA path.


In [ ]:
ft_job_id = step6_launch_finetuning(training_file_id)
state = load_state()
state.update({"fine_tune_job_id": ft_job_id, "last_completed_step": 6})
save_state(state)
ft_job_id


## Step 7 — Deploy the latest checkpoint as a private custom model

The deployment API validates the checkpoint against a serverless LoRA base model. This notebook mirrors the script by trying the fine-tune job model first and then falling back to the currently deployable base model candidates. A fresh adapter name is recommended on every retry.


In [ ]:
adapter_name = f"customer-support-notebook-{uuid.uuid4().hex[:6]}"
model_info = step7_deploy_lora(ft_job_id, adapter_name=adapter_name)
deployed_model_name = model_info["name"]
state = load_state()
state.update({
    "adapter_name": adapter_name,
    "deployed_model_name": deployed_model_name,
    "last_completed_step": 7,
})
save_state(state)
deployed_model_name


## Step 8 — Smoke test the deployed support assistant

A quick smoke test verifies that the deployed artifact behaves like a support assistant rather than merely existing as a successful job record. Look for concise, policy-aware answers and no fabricated promises.


In [ ]:
smoke_test_answers = step8_smoke_test(deployed_model_name)
state = load_state()
state.update({"last_completed_step": 8})
save_state(state)
smoke_test_answers


## Troubleshooting and reruns

- Re-open the saved state at any time with `load_state()`.
- If deployment fails because the job is not a LoRA job, rerun Step 6 with the current LoRA hyperparameters and then retry Step 7.
- If you retry deployment, use a fresh `adapter_name`.
- Inspect `customer_raw_dataset.jsonl`, `customer_batch_output.jsonl`, and `customer_curated_training.jsonl` in `scripts/artifacts/` when debugging data issues.
- You can adapt the same notebook pattern for billing, warranty, fraud, or internal IT helpdesk assistants by swapping the prompts and policy prompt.
